# Acceptance: calibration entry points built on the workflow pieces

**This is a sign-off page, not a tutorial.** It exercises roadmap step 30
(composability package `CP2`, findings `CX16`–`CX19`, `CX24`). `wf.optimize`
now returns a resumable `wf.OptimizeRun` and takes any optimisation method you
write; `BayesianModel.find_map` is a one-start `wf.optimize`;
`BayesianModel.sample` is a model-bound `wf.run_mcmc` that takes a numpyro
kernel you built and seeds from `Candidates`; both return the run object.
Nothing here asserts; the automated checks live in
`tests/test_calib_composable.py`.

Run every cell top to bottom (about a minute). Sign off only if every
**What to check** box holds.

| | You are checking that… |
|---|---|
| A | an optimisation run continues from where it stopped, and its history records every call |
| B | an optimisation method you write plugs into `wf.optimize` |
| C | `find_map` is exactly a one-start `wf.optimize`, and can keep going |
| D | `sample` uses the numpyro kernel you built and is exactly `wf.run_mcmc` |
| E | `sample` can sample until a stop rule decides |
| F | `posterior_runs` takes the run objects directly |

The model is the one-rate `S → I → R` of notebooks 19–22: the infection rate is
unknown (truth **0.35**), recovery is fixed.

In [ ]:
from typing import NamedTuple

import jax
import jax.numpy as jnp
import numpy as np
import optax
import pandas as pd
import plotly.io as pio
from numpyro.infer import NUTS

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TransitionFlow,
    derived_refs,
)
from summer4.epi.calibration import BayesianModel, NormalLikelihood, Uniform
from summer4.epi.calibration import workflow as wf


class Rates(NamedTuple):
    infection: float
    recovery: float


state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
refs = derived_refs(Rates)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], refs.infection))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], refs.recovery))
cm = model.compile()
y0 = PropertyData.wrap(pmap, np.array([999.0, 1.0, 0.0]))
times = np.array([0.0, 20.0, 40.0, 60.0])
infectious = Compartments(where=state["I"])
run_kwargs = {"t0": 0.0, "t1": 80.0, "dt": 1.0, "solver": "euler"}
truth = cm.run({"infection": 0.35, "recovery": 0.1}, y0, save=SavePlan(requests={"I": SaveRequest(infectious, ts=times)}), **run_kwargs)
observed = np.asarray(truth["I"].at_times(times).values.data).reshape(-1)
bm = BayesianModel(
    cm,
    {"recovery": 0.1},
    priors=(Uniform("infection", 0.05, 1.0),),
    targets=TargetSet(targets=(Target(key="I", times=times, values=observed, quantity=infectious, likelihood=NormalLikelihood(sd=5.0)),)),
    y0=y0,
    run_kwargs=run_kwargs,
)
starts = wf.evaluate(bm, wf.lhs(bm, 16, seed=1), batch_size=8).best(4)
starts.to_frame()

## A. An optimisation run continues from where it stopped

Adam from the four starts, with convergence switched off so every start runs
every step. One run takes 80 steps; a second takes 40, then `run.extend(40)`.
If `extend` really continues — optimiser state and all — the two loss traces
are the same curve. The plot shows each start's best loss after every chunk of
20 steps; the table compares the two runs and lists the stage history.

In [ ]:
adam = wf.Optax(learning_rate=0.05, plateau=False)
never_converge = wf.AutoTune(lr_grid=(0.05,), patience=10**6)
kw = dict(method=adam, tuning=never_converge, chunk_steps=20, seed=0)

one_run = wf.optimize(bm, starts, max_steps=80, **kw)
first = wf.optimize(bm, starts, max_steps=40, **kw)
first_trace = first.loss_trace.copy()
resumed = first.extend(40)

steps = 20 * np.arange(1, 5)
curves = {}
for i in range(len(starts)):
    curves[f"start {i}: one 80-step run"] = pd.Series(one_run.loss_trace[:, i], index=steps)
    curves[f"start {i}: 40 + extend(40)"] = pd.Series(resumed.loss_trace[:, i], index=steps)
fig = pd.DataFrame(curves).rename_axis("step").plot(markers=True, log_y=True, title="A. Best loss per start: one run vs a resumed run")
fig.update_layout(xaxis_title="optimiser steps", yaxis_title="best loss (-log density)", legend_title="")
fig.show()

pd.Series(
    {
        "chunks before extend": first_trace.shape[0],
        "chunks after extend": resumed.loss_trace.shape[0],
        "max |resumed - one run| (loss trace)": float(np.max(np.abs(resumed.loss_trace - one_run.loss_trace))),
        "max |resumed - one run| (best infection)": float(np.max(np.abs(resumed.candidates.params["infection"] - one_run.candidates.params["infection"]))),
        "history stages": [record.stage for record in resumed.history],
    },
    name="A",
    dtype=object,
)

> **What to check**
>
> - Each start's two markers (one run, resumed run) sit on top of each other at every step, including steps 60 and 80 after the `extend`.
> - The table shows 2 chunks before `extend` and 4 after, and both differences are 0 or at float32 rounding.
> - `history stages` lists the design stages (`lhs`, `evaluate`, …) followed by **two** `optimize` records — one for the first call, one for `extend`.

## B. An optimisation method you write plugs in

A method is anything with `make(potential_fn, ...)` returning a backend with
`init(z0, key)` and `step(state)`. This one is plain fixed-step gradient
descent — about twenty lines — and runs through the same chunked, vmapped,
restart-aware `wf.optimize` as the built-ins.

In [ ]:
class GradientDescent:
    def __init__(self, step_size):
        self.step_size = step_size

    def make(self, potential_fn, *, learning_rate=None, sites=None, template_z=None):
        step_size = self.step_size

        class Backend:
            def init(self, z0, key):
                z = {k: jnp.asarray(v) for k, v in z0.items()}
                return {"z": z, "best_z": z, "best_loss": potential_fn(z)}

            def step(self, state):
                loss, grads = jax.value_and_grad(potential_fn)(state["z"])
                z = jax.tree.map(lambda v, g: v - step_size * g, state["z"], grads)
                better = loss < state["best_loss"]
                best_z = jax.tree.map(lambda cur, best: jnp.where(better, cur, best), state["z"], state["best_z"])
                return {"z": z, "best_z": best_z, "best_loss": jnp.where(better, loss, state["best_loss"])}, best_z, state["best_loss"]

        return Backend()


mine = wf.optimize(bm, starts, method=GradientDescent(2e-4), tuning=never_converge, chunk_steps=20, max_steps=200)
trace = pd.DataFrame(mine.loss_trace, columns=[f"start {i}" for i in range(len(starts))], index=20 * np.arange(1, mine.loss_trace.shape[0] + 1))
fig = trace.rename_axis("step").plot(markers=True, log_y=True, title="B. Your gradient descent, run by wf.optimize")
fig.update_layout(xaxis_title="optimiser steps", yaxis_title="best loss (-log density)", legend_title="")
fig.show()
pd.Series(
    {
        "isinstance(method, wf.OptimizeMethod)": isinstance(GradientDescent(1.0), wf.OptimizeMethod),
        "run.method_name": mine.method_name,
        "best infection (truth 0.35)": round(float(mine.best_params["infection"]), 3),
    },
    name="B",
    dtype=object,
)

> **What to check**
>
> - Every start's loss falls (or stays flat once it has arrived) across the 200 steps.
> - The table shows `True`, `gradientdescent`, and a best infection rate close to 0.35.

## C. `find_map` is a one-start `wf.optimize`

`find_map` returns the optimisation run; `best_params` is the MAP estimate as a
plain dict. Its docstring lists the lines it stands for; the cell prints them,
runs them by hand, and compares. Then `run.extend(200)` keeps optimising from
where `find_map` stopped.

In [ ]:
doc = bm.find_map.__doc__
print(doc[doc.index("Exactly::"):])

mapped = bm.find_map(steps=40, seed=2)
z0 = bm.init_point(2)
start = wf.Candidates.from_z(bm, {k: jnp.asarray(v)[None] for k, v in z0.items()})
by_hand = wf.optimize(bm, start, method=wf.Optax(optax.adam(0.05)), max_steps=40, chunk_steps=40, seed=2)
after_40 = float(mapped.best_params["infection"])
log_density_40 = float(mapped.candidates.log_density[0])
mapped.extend(200)

pd.Series(
    {
        "type": type(mapped).__name__,
        "best infection after 40 steps": round(after_40, 4),
        "same by hand": round(float(by_hand.best_params["infection"]), 4),
        "log density after 40 steps": round(log_density_40, 3),
        "log density after extend(200)": round(float(mapped.candidates.log_density[0]), 3),
        "best infection after extend (truth 0.35)": round(float(mapped.best_params["infection"]), 4),
    },
    name="C",
    dtype=object,
)

> **What to check**
>
> - The printed expansion is `init_point` → `Candidates.from_z` → `wf.optimize(..., method=wf.Optax(optimizer or optax.adam(0.05)), ...)`.
> - `type` is `OptimizeRun`, and `best infection after 40 steps` equals `same by hand` exactly.
> - The log density after `extend(200)` is at least as high as after 40 steps, and the infection rate is then close to 0.35.

## D. `sample` uses the kernel you built

Build a NUTS kernel yourself — here with `target_accept_prob=0.95`, a setting
`sample` never had a keyword for — and hand it to `sample` with four chains
seeded from the design's best starts. The run object holds your kernel. The
same draws are then produced by `wf.run_mcmc`, the lines `sample` stands for.

In [ ]:
kernel = NUTS(bm.numpyro_model(), target_accept_prob=0.95)
sampled = bm.sample(kernel, init=starts, num_warmup=100, num_samples=100, num_chains=4, seed=7)
again = wf.run_mcmc(
    bm, starts, make_kernel=lambda: NUTS(bm.numpyro_model(), target_accept_prob=0.95),
    num_chains=4, num_warmup=100, chunk_samples=100, stop=lambda row: (True, "num_samples"), seed=7,
)

draws = pd.DataFrame(sampled.samples["infection"].T, columns=[f"chain {c}" for c in range(4)]).rename_axis("draw")
fig = draws.plot(title="D. Four seeded chains from your NUTS kernel")
fig.add_hline(y=0.35, line_dash="dash", annotation_text="truth")
fig.update_layout(xaxis_title="draw", yaxis_title="infection rate", legend_title="")
fig.show()

pd.Series(
    {
        "type": type(sampled).__name__,
        "run.mcmc.sampler is your kernel": sampled.mcmc.sampler is kernel,
        "kernel target_accept_prob": kernel._target_accept_prob,
        "draws per chain": sampled.samples["infection"].shape[1],
        "max |sample - run_mcmc|": float(np.max(np.abs(sampled.samples["infection"] - again.samples["infection"]))),
        "run.idata posterior shape": tuple(np.asarray(sampled.idata.posterior["infection"]).shape),
    },
    name="D",
    dtype=object,
)

> **What to check**
>
> - The four chains overlap and wander around the dashed truth line.
> - `type` is `MCMCRun`, `run.mcmc.sampler is your kernel` is **True**, and the kernel's `target_accept_prob` reads **0.95**.
> - `max |sample - run_mcmc|` is exactly **0.0**, and `run.idata` holds 4 chains × 100 draws.

## E. `sample` can run until a stop rule decides

With `stop=` a `wf.StopRule`, `num_samples` becomes the chunk size and `sample`
keeps drawing until R-hat ≤ 1.05 and ESS ≥ 200 (at most 2,000 draws per chain).
The progress table is the run's own record of each decision.

In [ ]:
ruled = bm.sample(init=starts, num_chains=4, num_warmup=100, num_samples=50, stop=wf.StopRule(rhat=1.05, ess=200, max_samples=2000), seed=0)
fig = ruled.progress.plot(y="ess_bulk_min", markers=True, title="E. Smallest bulk ESS after each chunk")
fig.add_hline(y=200, line_dash="dash", annotation_text="StopRule.ess")
fig.update_layout(xaxis_title="chunk", yaxis_title="min bulk ESS", showlegend=False)
fig.show()
ruled.progress

> **What to check**
>
> - `decision` is empty on every row except the last, which reads `diagnostics`.
> - On the last row `rhat_max` ≤ 1.05 and `ess_bulk_min` ≥ 200; on every earlier row at least one of the two fails.

## F. `posterior_runs` takes the run objects

`posterior_runs` accepts the `MCMCRun` itself (its draws) and the `OptimizeRun`
itself (its best points), as well as the `InferenceData` it took before. The
first two ribbons below come from the same draws, once passed as the run and
once as `run.idata`; the markers are the four optimised starts' trajectories.

In [ ]:
from_run = bm.posterior_runs(sampled, n=60, seed=1, batch_size=20)
from_idata = bm.posterior_runs(sampled.idata, n=60, seed=1, batch_size=20)
from_optimum = bm.posterior_runs(one_run, n=None, batch_size=4)

ribbon = from_run.quantiles(q=(0.025, 0.5, 0.975))["baseline"]
band = pd.DataFrame(
    {
        "2.5% (MCMCRun)": ribbon[("I", "0.025")].to_numpy(),
        "median (MCMCRun)": ribbon[("I", "0.5")].to_numpy(),
        "97.5% (MCMCRun)": ribbon[("I", "0.975")].to_numpy(),
    },
    index=np.asarray(ribbon.index),
).rename_axis("time")
fig = band.plot(title="F. Posterior ribbon from the run, with the optimised starts")
for i, path in enumerate(from_optimum.samples("baseline", "I")):
    fig.add_scatter(x=band.index, y=np.asarray(path).reshape(-1), mode="markers", name=f"optimised start {i}", marker={"size": 4})
fig.update_layout(xaxis_title="time", yaxis_title="infectious", legend_title="")
fig.show()

pd.Series(
    {
        "max |from run - from idata|": float(np.max(np.abs(from_run.samples("baseline", "I") - from_idata.samples("baseline", "I")))),
        "optimised trajectories": from_optimum.samples("baseline", "I").shape[0],
    },
    name="F",
    dtype=object,
)

> **What to check**
>
> - The ribbon is ordered (2.5% ≤ median ≤ 97.5%) and the four optimised trajectories sit inside it.
> - `max |from run - from idata|` is exactly **0.0**: passing the run is the same as passing `run.idata`.
> - There are **4** optimised trajectories, one per start of section A's run.